<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%209/04_Part_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Execute this cell to import data, run the GMM, and generate visualizations
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from google.colab import files
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# --- Step 1: Secure Dataset Ingestion ---
csv_filename = "CC GENERAL.csv"
if not os.path.exists(csv_filename):
    print(f"⚠️ {csv_filename} not found in environment directory.")
    print("Please select and upload your downloaded 'CC GENERAL.csv' file below:")
    uploaded = files.upload()

# Read the uploaded dataset
df = pd.read_csv(csv_filename)

# Select targeted features and drop missing entries to maintain data integrity
features = ['PURCHASES', 'CREDIT_LIMIT']
data_clean = df[features].dropna().values

# --- Step 2: Object-Oriented GMM Class Framework ---
class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(
            n_components=self.n_components,
            covariance_type='full',
            random_state=self.random_state
        )

    def prepare_data(self, X):
        # Split into training (80%) and validation/test (20%) sets
        X_train, X_test = train_test_split(X, test_size=0.20, random_state=self.random_state)
        # Apply standard scaling based on training statistics to mitigate feature magnitude bias
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        return X_train_scaled, X_test_scaled

    def fit_model(self, X_train):
        self.gmm.fit(X_train)
        print("="*50)
        print("          EXPECTATION-MAXIMIZATION SUMMARY          ")
        print("="*50)
        print(f"Convergence Status:       {self.gmm.converged_}")
        print(f"EM Iterations Required:   {self.gmm.n_iter_}")

    def evaluate_generalization(self, X_test):
        # Compute the average log-likelihood on unseen test data
        avg_log_likelihood = self.gmm.score(X_test)
        print(f"Test Set Log-Likelihood:  {avg_log_likelihood:.4f}")
        print("="*50)

    def plot_raw_density(self, X_train):
        # Limit axis viewing range slightly to reduce visual distortion from extreme outliers
        q_x = np.percentile(X_train[:, 0], 99)
        q_y = np.percentile(X_train[:, 1], 99)
        mask = (X_train[:, 0] <= q_x) & (X_train[:, 1] <= q_y)

        fig = px.density_heatmap(
            x=X_train[mask, 0], y=X_train[mask, 1],
            marginal_x="histogram", marginal_y="histogram",
            nbinsx=40, nbinsy=40,
            color_continuous_scale="Viridis",
            labels={'x': 'Standardized Purchases', 'y': 'Standardized Credit Limit'}
        )
        fig.update_layout(
            title="Figure 1: Empirical 2D Density Heatmap with Marginal Distributions",
            template="plotly_white"
        )
        fig.show()

    def generate_mesh_responsibilities(self, X_train):
        # Generate a dense evaluation mesh across the sample domain space
        x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
        y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        mesh_points = np.c_[xx.ravel(), yy.ravel()]

        # Compute exact conditional responsibilities gamma_ik via the E-step equations
        resp_mesh = self.gmm.predict_proba(mesh_points)
        # Select the maximum posterior probability value for continuous background contouring
        max_resp = resp_mesh.max(axis=1).reshape(xx.shape)
        return xx, yy, max_resp

    def plot_assignments(self, X_data, title_string):
        xx, yy, max_resp = self.generate_mesh_responsibilities(X_data)
        # Hard assignments for the scattered overlay markers
        labels = self.gmm.predict(X_data)

        fig = go.Figure()

        # Add the background contour representing the strength of the expectation vector
        fig.add_trace(go.Contour(
            x=np.linspace(xx.min(), xx.max(), 200),
            y=np.linspace(yy.min(), yy.max(), 200),
            z=max_resp,
            colorscale="Electric",
            contours_coloring="heatmap",
            line_width=0,
            colorbar=dict(title="max(γ_ik)")
        ))

        # Overlay the discrete data observations
        fig.add_trace(go.Scatter(
            x=X_data[:, 0], y=X_data[:, 1],
            mode='markers',
            marker=dict(color=labels, colorscale="Plotly3", size=5, line=dict(width=0.5, color='white')),
            name='Observations'
        ))

        fig.update_layout(
            title=title_string,
            xaxis_title="Standardized Purchases",
            yaxis_title="Standardized Credit Limit",
            template="plotly_white"
        )
        fig.show()

# --- Step 3: Pipeline Orchestration Execution ---
segmenter = GMMFinancialSegmenter(n_components=3)
X_train, X_test = segmenter.prepare_data(data_clean)

# Execute core algorithms
segmenter.fit_model(X_train)
segmenter.evaluate_generalization(X_test)

# Render structural interactive plots
segmenter.plot_raw_density(X_train)
segmenter.plot_assignments(X_train, "Figure 2: GMM Training Assignment Space & Posterior Contours")
segmenter.plot_assignments(X_test, "Figure 3: GMM Out-of-Sample Test Validation Space")